# Notebook 04: Gold atomic Finance fact
Reads the current ACDOCA and BKPF state from upserted Silver, preserves one ACDOCA document-line grain, enriches from BKPF, resolves point-in-time SCD2 dimension keys, and performs an idempotent Gold MERGE on the deterministic finance transaction key.

In [ ]:
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable
from functools import reduce
import datetime
import uuid

BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
SLV = f"{BRZ_SLV_LH}.slv"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
HIGH_DATE = "9999-12-31"
BATCH_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)

In [ ]:
# Silver stores one current row per business key after full-snapshot MERGE.
ACDOCA_KEYS=["RCLNT","RLDNR","RBUKRS","GJAHR","BELNR","DOCLN"]
BKPF_KEYS=["MANDT","BUKRS","GJAHR","BELNR"]

def dim_join(df,dim_name,alias,conditions,date_col="POSTING_DATE"):
    dim=spark.table(f"{DIM}.{dim_name}").alias(alias)
    cond=reduce(lambda a,b:a&b,conditions)&(F.col(date_col)>=F.col(f"{alias}.EFFECTIVE_FROM"))&(F.col(date_col)<=F.col(f"{alias}.EFFECTIVE_TO"))
    return df.join(dim,cond,"left")

ac=spark.table(f"{SLV}.acdoca").alias("a")
bk=spark.table(f"{SLV}.bkpf").alias("h")
# Header enrichment stays optional and must not alter the ACDOCA line grain.
joined=ac.join(bk,(F.col("a.RCLNT").eqNullSafe(F.col("h.MANDT")))&(F.col("a.RBUKRS").eqNullSafe(F.col("h.BUKRS")))&(F.col("a.GJAHR").eqNullSafe(F.col("h.GJAHR")))&(F.col("a.BELNR").eqNullSafe(F.col("h.BELNR"))),"left")

def a(name,typ="string"): return F.col(f"a.{name}") if name in ac.columns else F.lit(None).cast(typ)
def h(name,typ="string"): return F.col(f"h.{name}") if name in bk.columns else F.lit(None).cast(typ)

fact=joined.withColumn("POSTING_DATE",F.coalesce(a("BUDAT","date"),h("BUDAT","date"),F.to_date(F.concat_ws("-",a("GJAHR"),F.lpad(a("POPER"),2,"0"),F.lit("01")))))
fact=dim_join(fact,"company","co",[a("RCLNT").eqNullSafe(F.col("co.MANDT")),a("RBUKRS").eqNullSafe(F.col("co.BUKRS"))])
fact=dim_join(fact,"gl_account","gl",[a("RCLNT").eqNullSafe(F.col("gl.MANDT")),F.col("co.KTOPL").eqNullSafe(F.col("gl.KTOPL")),a("RACCT").eqNullSafe(F.col("gl.SAKNR"))])
fact=dim_join(fact,"cost_center","cc",[a("RCLNT").eqNullSafe(F.col("cc.MANDT")),a("KOKRS").eqNullSafe(F.col("cc.KOKRS")),a("RCNTR").eqNullSafe(F.col("cc.KOSTL"))])
fact=dim_join(fact,"profit_center","pc",[a("RCLNT").eqNullSafe(F.col("pc.MANDT")),a("KOKRS").eqNullSafe(F.col("pc.KOKRS")),a("PRCTR").eqNullSafe(F.col("pc.PRCTR"))])
fact=dim_join(fact,"division","dv",[F.col("pc.KHINR").eqNullSafe(F.col("dv.DIVISION_CODE"))])
fact=dim_join(fact,"vendor","ve",[a("RCLNT").eqNullSafe(F.col("ve.MANDT")),a("LIFNR").eqNullSafe(F.col("ve.LIFNR"))])
fact=dim_join(fact,"customer","cu",[a("RCLNT").eqNullSafe(F.col("cu.MANDT")),a("KUNNR").eqNullSafe(F.col("cu.KUNNR"))])

out=fact.select(
 F.sha2(F.concat_ws("||",*[F.coalesce(a(c).cast("string"),F.lit("__NULL__")) for c in ACDOCA_KEYS]),256).alias("FINANCE_TRANSACTION_SK"),
 F.date_format("POSTING_DATE","yyyyMMdd").cast("int").alias("POSTING_DATE_SK"),
 ((a("GJAHR","int") * F.lit(100)) + a("POPER","int")).cast("int").alias("FISCAL_PERIOD_KEY"),
 F.col("co.DIMENSION_SK").alias("COMPANY_SK"),F.col("gl.DIMENSION_SK").alias("GL_ACCOUNT_SK"),F.col("cc.DIMENSION_SK").alias("COST_CENTER_SK"),F.col("pc.DIMENSION_SK").alias("PROFIT_CENTER_SK"),F.col("dv.DIMENSION_SK").alias("DIVISION_SK"),F.col("ve.DIMENSION_SK").alias("VENDOR_SK"),F.col("cu.DIMENSION_SK").alias("CUSTOMER_SK"),
 *[a(c).alias(c) for c in ["RCLNT","RLDNR","RBUKRS","GJAHR","POPER","BELNR","DOCLN","RACCT","RCNTR","PRCTR","KOKRS","LIFNR","KUNNR","DRCRK","BLART","AWTYP","SGTXT","RHCUR","RKCUR","RWCUR"]],
 h("BKTXT").alias("DOCUMENT_HEADER_TEXT"),h("STBLG").alias("REVERSAL_DOCUMENT"),
 a("HSL","decimal(23,2)").alias("AMOUNT_LOCAL"),a("KSL","decimal(23,2)").alias("AMOUNT_GROUP"),a("TSL","decimal(23,2)").alias("AMOUNT_TRANSACTION"),a("WSL","decimal(23,2)").alias("AMOUNT_OTHER"),a("MSL","decimal(23,3)").alias("QUANTITY"),
 a("BATCH_RUN_ID").alias("SOURCE_BATCH_RUN_ID"),a("EXTRACT_DATE","date").alias("EXTRACT_DATE"),a("LOAD_DATE","date").alias("SILVER_LOAD_DATE"),a("SOURCE_SYSTEM").alias("SOURCE_SYSTEM"),a("SOURCE_FILE_NAME").alias("SOURCE_FILE_NAME"),F.lit(BATCH_ID).alias("GOLD_BATCH_ID"),F.current_timestamp().alias("GOLD_LOAD_TIMESTAMP"))

# Idempotent Gold merge. This is Gold curation, not Silver historization.
target=f"{FCT}.finance_transaction"
if spark.catalog.tableExists(target):
    (DeltaTable.forName(spark,target).alias("tgt").merge(out.alias("src"),"tgt.FINANCE_TRANSACTION_SK = src.FINANCE_TRANSACTION_SK").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    out.write.format("delta").mode("overwrite").saveAsTable(target)

# Reconciliation at ACDOCA line grain.
source_count=ac.count(); output_count=out.select("FINANCE_TRANSACTION_SK").distinct().count(); duplicate_count=out.groupBy("FINANCE_TRANSACTION_SK").count().filter(F.col("count")>1).count()
recon=spark.createDataFrame([(BATCH_ID,source_count,output_count,duplicate_count)],"gold_batch_id string, source_line_count long, output_distinct_count long, duplicate_key_count long").withColumn("gold_load_timestamp",F.current_timestamp())
recon.write.mode("append").format("delta").saveAsTable(f"{AUD}.finance_fact_reconciliation")
display(recon)